<a href="https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [12]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Subhash-2910/flyrank-ML-T1.git"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ML-T1/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [13]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I chose **Logistic Regression** for my Refresh / Content Opportunity Scoring lane.

The model predicts whether a page’s observed trend direction is `down`, using historical content, search, visibility, position, and engagement signals. Logistic Regression is appropriate because it is transparent, fast, and allows me to inspect which signals are associated with a higher decline-risk score.

I will compare it with my Week-4 stale-and-visible baseline on the same held-out clients and with the same ranking metric, Precision@20.

This model supports human review. It does not prove that refreshing a page will cause improved performance.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

# Target: pages with an observed downward trend.
target_col = "decline_label"
df[target_col] = (df["trend_direction"] == "down").astype(int)

# Group only: never use client_id or content_id as model features.
group_col = "client_id"
id_col = "content_id"

# Historical numeric signals selected for the model.
numeric_features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
]

# Stable content/context categories. Tiers are excluded because they duplicate
# the continuous measurements above.
categorical_features = [
    "competition_level",
    "content_type",
    "main_intent",
]

feature_cols = numeric_features + categorical_features

# Do not use outcome, target-derived fields, comparison-window fields, IDs, or tiers.
excluded_from_features = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "age_tier",
    "age_tier_order",
    "freshness_tier",
    "word_count_tier",
    "char_count_tier",
    "impression_tier",
    "position_tier",
    "provider_used",
    "model_used",
]

model_df = df[[id_col, group_col, target_col] + feature_cols].copy()

print("Rows:", len(model_df))
print("Clients:", model_df[group_col].nunique())
print("Observed downward-trend rate:", f"{model_df[target_col].mean():.1%}")
print("Features used:", len(feature_cols))

Rows: 30000
Clients: 32
Observed downward-trend rate: 54.2%
Features used: 25


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*





I use an 80/20 client-grouped split. All pages from a client are assigned either to training or to testing, never both.

This is more honest than a random page split because pages from the same client can share content strategy, traffic patterns, and measurement practices. Grouping by client tests whether the model can generalize to unseen clients.

The outcome is whether the observed `trend_direction` is `down`. The 30-day comparison columns and `trend_pct` are excluded from features because they are used to create that outcome.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        y=model_df[target_col],
        groups=model_df[group_col],
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

train_clients = set(train_df[group_col])
test_clients = set(test_df[group_col])

assert train_clients.isdisjoint(test_clients), "Client overlap found."

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", train_df[group_col].nunique())
print("Test clients:", test_df[group_col].nunique())
print("Client overlap: none")
print("Train downward-trend rate:", f"{train_df[target_col].mean():.1%}")
print("Test downward-trend rate:", f"{test_df[target_col].mean():.1%}")

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: none
Train downward-trend rate: 55.0%
Test downward-trend rate: 51.1%


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*





I train Logistic Regression on the training clients and score the held-out test clients.

The comparison is fair because the Week-4 baseline and the model are ranked on the same held-out clients and evaluated with the same metric: Precision@20.

The Week-4 baseline gives 60% weight to content age and 40% weight to current 90-day impressions. The model can combine those signals with additional historical search, position, engagement, content, and competition signals.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score

X_train = train_df[feature_cols]
y_train = train_df[target_col]

X_test = test_df[feature_cols]
y_test = test_df[target_col]

# Numeric data is median-imputed and scaled.
# Categorical data is filled and one-hot encoded.
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("onehot", OneHotEncoder(handle_unknown="ignore")),
            ]),
            categorical_features,
        ),
    ]
)

model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "logistic_regression",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

model.fit(X_train, y_train)

# Model ranking score on held-out clients.
test_results = test_df[[id_col, group_col, target_col]].copy()
test_results["model_score"] = model.predict_proba(X_test)[:, 1]

# Rebuild the Week-4 stale-and-visible score on the SAME held-out test set.
# Higher age and higher impressions create a higher review score.
test_results["baseline_score"] = (
    0.60 * test_df["content_age_days"].rank(pct=True)
    + 0.40 * test_df["impressions_90d"].rank(pct=True)
)

def precision_at_k(frame, score_col, label_col, k=20):
    top_k = frame.nlargest(k, score_col)
    return top_k[label_col].mean()

comparison = pd.DataFrame({
    "approach": [
        "Week-4 stale-and-visible baseline",
        "Logistic Regression",
    ],
    "precision_at_20": [
        precision_at_k(test_results, "baseline_score", target_col, k=20),
        precision_at_k(test_results, "model_score", target_col, k=20),
    ],
    "average_precision": [
        average_precision_score(y_test, test_results["baseline_score"]),
        average_precision_score(y_test, test_results["model_score"]),
    ],
    "roc_auc": [
        roc_auc_score(y_test, test_results["baseline_score"]),
        roc_auc_score(y_test, test_results["model_score"]),
    ],
}).round(3)

display(comparison)

print(
    "Model Precision@20:",
    f"{precision_at_k(test_results, 'model_score', target_col, k=20):.1%}"
)
print(
    "Baseline Precision@20:",
    f"{precision_at_k(test_results, 'baseline_score', target_col, k=20):.1%}"
)

,approach,precision_at_20,average_precision,roc_auc
0,Week-4 stale-and-visible baseline,0.45,0.476,0.473
1,Logistic Regression,0.80,0.597,0.596


Model Precision@20: 80.0%
Baseline Precision@20: 45.0%


## Comparison interpretation

The table compares both approaches on the same held-out clients.

I will use the measured Precision@20 result as the main decision metric because this project is a prioritization task: the important question is how many genuinely downward-trending pages appear near the top of the review queue.

The Logistic Regression model is only worthwhile if its measured ranking performance improves enough over the transparent previous week baseline to justify the additional complexity.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*



I inspect false positives because they are pages the model prioritizes for review even though their observed trend direction was not `down`.

These errors can happen because SEO performance is affected by seasonality, search-intent changes, competition, and client-specific context that are not fully represented by the available features.

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature interpretation: largest absolute Logistic Regression coefficients.
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coefficients = model.named_steps["logistic_regression"].coef_[0]

feature_interpretation = (
    pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefficients,
        "absolute_coefficient": np.abs(coefficients),
    })
    .sort_values("absolute_coefficient", ascending=False)
    .head(15)
    .round(3)
)

print("Most influential model features:")
display(feature_interpretation)

# Use a transparent threshold only for reviewing classification errors.
test_results["predicted_down"] = (test_results["model_score"] >= 0.50).astype(int)

false_positives = (
    test_results[
        (test_results["predicted_down"] == 1)
        & (test_results[target_col] == 0)
    ]
    .sort_values("model_score", ascending=False)
    .head(10)
)

false_negatives = (
    test_results[
        (test_results["predicted_down"] == 0)
        & (test_results[target_col] == 1)
    ]
    .sort_values("model_score")
    .head(10)
)

print("Top false positives:")
display(false_positives)

print("False negatives:")
display(false_negatives)

Most influential model features:


,feature,coefficient,absolute_coefficient
9,numeric__users_90d,-1.046,1.046
8,numeric__sessions_90d,0.834,0.834
13,numeric__days_with_impressions,0.601,0.601
30,categorical__main_intent_navigational,-0.464,0.464
14,numeric__days_with_sessions,-0.422,0.422
15,numeric__content_age_days,-0.402,0.402
26,categorical__content_type_feedly article,-0.391,0.391
27,categorical__content_type_keyword article,0.343,0.343
12,numeric__scroll_events_90d,0.264,0.264
3,numeric__word_count,0.252,0.252


Top false positives:


,content_id,client_id,decline_label,model_score,baseline_score,predicted_down
10175,content_374e795aab68,client_f369cb89fc,0,0.889199,0.383985,1
8016,content_c94a53e3bfb8,client_f369cb89fc,0,0.877212,0.314668,1
26614,content_7be5f150dc65,client_f369cb89fc,0,0.872934,0.211228,1
27993,content_26d48a980581,client_f369cb89fc,0,0.867203,0.314084,1
6903,content_c84a0ab98e90,client_f369cb89fc,0,0.838204,0.422018,1
17602,content_3b26815717ae,client_f369cb89fc,0,0.837500,0.370047,1
11202,content_ea1fdec27b19,client_f369cb89fc,0,0.836715,0.373503,1
16052,content_07d062db2b51,client_f369cb89fc,0,0.835044,0.291433,1
20295,content_31bd7e88b345,client_f369cb89fc,0,0.832051,0.257277,1
4397,content_53a574b8a7da,client_f369cb89fc,0,0.826260,0.284099,1


False negatives:


,content_id,client_id,decline_label,model_score,baseline_score,predicted_down
17127,content_8818fd6d967f,client_4e07408562,1,0.049726,0.909022,0
21819,content_4c36c775b818,client_4e07408562,1,0.091667,0.848239,0
1010,content_8ff857ae67d0,client_e629fa6598,1,0.095454,0.864579,0
26413,content_bdd7c88a58ed,client_4e07408562,1,0.101374,0.822083,0
24849,content_2f002563e9cd,client_e629fa6598,1,0.104131,0.623446,0
19701,content_57971022aadc,client_4e07408562,1,0.110705,0.988382,0
27487,content_31c66d071a62,client_8527a891e2,1,0.113228,0.331235,0
12064,content_4090f0acd977,client_4e07408562,1,0.116372,0.819779,0
23511,content_4de8c62603bf,client_e629fa6598,1,0.123554,0.539737,0
23523,content_38b9529bf1b6,client_e629fa6598,1,0.125859,0.651290,0


## Interpretation

The coefficient table shows which inputs the Logistic Regression relied on most strongly. A positive coefficient is associated with a higher downward-trend score; a negative coefficient is associated with a lower score. These are associations in this dataset, not causal effects.

The false-positive table shows pages that the model ranked as downward-trend candidates but that did not have an observed `down` outcome. These may reflect missing context, seasonality, or client-specific patterns.

The model should remain a decision-support tool. A high score means “review this page first,” not “refreshing this page will certainly improve performance.”

In [18]:
assert set(train_df[group_col]).isdisjoint(set(test_df[group_col]))
assert "trend_pct" not in feature_cols
assert "trend_direction" not in feature_cols
assert "content_id" not in feature_cols
assert "client_id" not in feature_cols

print("Self-check passed.")
print("Model-versus-baseline comparison completed on the same held-out clients.")

Self-check passed.
Model-versus-baseline comparison completed on the same held-out clients.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.